# AI Labor Market Intelligence — دفتر التدريب النهائي (AraBERT + LSTM)

هذا الدفتر مبني على طريقة `LMI_Drive_Run.ipynb` التي أرسلتها، وموجه لنسخ مشروع **LMI_PROJECT** الكاملة.

**ما سيدربه فعلياً:** AraBERT لنوع الفرصة، AraBERT للنطاق الجغرافي، AraBERT NER، وأربعة نماذج LSTM للمهارات والتخصصات والمسميات والمحافظات (إجمالي **7 مهام تدريب**).

**ضروري قبل التشغيل:** ضع **المشروع الكامل** في `MyDrive/LMI_PROJECT/`، أو ملف `LMI_PROJECT_V3_VERIFIED.zip` / `LMI_PROJECT.zip` في جذر Google Drive. ملف `LMI_GEO_NEURAL_V3_PATCH.zip` **ليس مشروعاً كاملاً** ولا يكفي بمفرده. يحتاج تدريب AraBERT إلى اتصال بالإنترنت لتنزيل الأوزان الأصلية، ويُفضَّل GPU. يمكن تشغيل LSTM على CPU.

**تنبيه:** يحتوي الدفتر على نسخة مضغوطة من *ملفات التدريب والمرافق المتوافقة من المشروع الذي تمت مراجعته*. تُطبَّق فقط على نسخة العمل المؤقتة في `/content/lmi`؛ **لا تُستبدل أكواد مشروعك الأصلية في Drive**. يتم حفظ الأوزان والتقارير في Drive.

النتائج لا تُكتب مسبقاً: أي قياسات ظاهرة بعد التشغيل تُحسب فعلياً. تقييم Silver ليس Gold بشرياً. لا يمكن ضمان توفر GPU أو الإنترنت أو عدم انقطاع جلسة Colab.

## 1) خيارات التشغيل — اضبطها هنا فقط

القيم الافتراضية تعيد **تدريب المهام السبع**. للتشغيل الثاني دون تكرار التدريب غيّر الوضع إلى `missing`؛ وسيُتخطى النموذج فقط عند اكتمال ملفاته.

In [ ]:
PROJECT_NAME = "LMI_PROJECT"
ARABERT_MODE = "force"       # force: إعادة التدريب | missing: تدريب الناقص فقط
LSTM_MODE = "force"          # force: تدريب العائلات الأربع | missing: تدريب الناقص فقط
ARABERT_TASKS = ("opportunity_type", "geographic_scope", "ner")
LSTM_FAMILIES = ("skills", "specializations", "job_titles", "governorates")
AS_OF = "2026-11-08"          # يُستبعد شهر نوفمبر غير المكتمل في هذه النسخة
CPU_THREADS = 2
USE_VERIFIED_TRAINING_CODE = True  # إصلاح توافق سكربتات التدريب محلياً؛ لا يغير مشروع Drive
RUN_TESTS = True
RUN_STREAMLIT = False          # اختياري، ليس ضرورياً للتدريب
PACKAGE_FOR_WINDOWS = False    # اختياري: يجمع النماذج والتقارير بعد التدريب

assert ARABERT_MODE in ("force", "missing")
assert LSTM_MODE in ("force", "missing")
from datetime import date
date.fromisoformat(AS_OF)
print("تم تثبيت إعدادات التدريب. AraBERT:", ARABERT_MODE, "| LSTM:", LSTM_MODE)

## 2) ربط Google Drive وتحضير مصدر المشروع الكامل

إذا لم يوجد مجلد `LMI_PROJECT`، يبحث الدفتر عن ZIP كامل ويستخرجه **بأسماء ملفات آمنة**. لا يحاول اعتبار حزمة التصحيح الجغرافي مشروعاً كاملاً.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, io, json, shutil, zipfile, subprocess, time, traceback, hashlib

MY = Path('/content/drive/MyDrive')
PROJ = MY / PROJECT_NAME
WORK = Path('/content/lmi')

def is_project(p):
    return all((p / rel).is_file() for rel in (
        'src/common.py', 'config/training.json',
        'src/training/train_classifier.py', 'src/training/train_ner.py',
        'src/forecasting/train_lstm.py',
        'FINAL_TRAINING_DATASET/unique_opportunities.csv'))

def safely_unpack_project(zip_path, target):
    with zipfile.ZipFile(zip_path) as z:
        files = [x for x in z.infolist() if not x.is_dir()]
        names = [x.filename.replace('\\', '/') for x in files]
        if any(x.startswith('/') or '..' in Path(x).parts for x in names):
            raise ValueError('ZIP غير آمن: مسار خارج مجلد المشروع')
        candidates = [n[:-len('src/common.py')] for n in names if n.endswith('src/common.py')]
        prefix = next((pre for pre in candidates if pre + 'config/training.json' in names
                       and pre + 'FINAL_TRAINING_DATASET/unique_opportunities.csv' in names), None)
        if prefix is None:
            raise FileNotFoundError('ZIP لا يحتوي على مشروع LMI كامل وبيانات FINAL_TRAINING_DATASET')
        target.mkdir(parents=True, exist_ok=True)
        for info, n in zip(files, names):
            if not n.startswith(prefix):
                continue
            rel = Path(n[len(prefix):])
            if not rel.parts or '..' in rel.parts:
                continue
            out = target / rel
            out.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as src, open(out, 'wb') as dst:
                shutil.copyfileobj(src, dst)

if not is_project(PROJ):
    candidates = [MY / x for x in ('LMI_PROJECT_V3_VERIFIED.zip', 'LMI_PROJECT.zip',
                                   'LMI_PROJECT_ENGINEERED.zip', 'LMI_PROJECT_REPAIRED.zip')]
    candidates += [Path('/content') / x.name for x in candidates]
    selected = next((x for x in candidates if x.is_file()), None)
    if selected is None:
        raise FileNotFoundError(
            'لم أجد مشروع LMI كاملاً. ارفع LMI_PROJECT_V3_VERIFIED.zip أو LMI_PROJECT.zip '
            'إلى MyDrive، أو ضع مجلد LMI_PROJECT الكامل فيها. حزمة GEO_PATCH وحدها لا تكفي.')
    print('فك المشروع الكامل من:', selected)
    safely_unpack_project(selected, PROJ)
assert is_project(PROJ), 'لا يزال المشروع غير مكتمل بعد استخراج ZIP'
print('✓ المشروع الكامل:', PROJ)

## 3) تثبيت الاعتمادات اللازمة وفحص البيئة

يعتمد الدفتر على PyTorch الموجود في Colab، ويثبت مكتبات AraBERT وNER والتقييم. لا تستعمل الإصدارات المثبتة سابقاً إذا كانت قديمة وغير متوافقة.

In [ ]:
import subprocess, sys, importlib.util
packages = [
    'transformers>=4.47,<4.56', 'accelerate>=0.34,<2',
    'safetensors>=0.4', 'arabert', 'seqeval>=1.2',
    'scikit-learn>=1.3', 'sklearn-crfsuite', 'matplotlib',
    'streamlit>=1.40,<2', 'plotly>=5.20,<7', 'pytest', 'joblib'
]
print('Installing training dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
import torch, pandas as pd, numpy as np, transformers
from arabert.preprocess import ArabertPreprocessor
from seqeval.metrics import f1_score
print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__,
      '| transformers:', transformers.__version__, '| pandas:', pd.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'غير متاح (AraBERT على CPU بطيء)')

## 4) نسخ المشروع إلى `/content/lmi` وإصلاح ملفات التدريب المتوافقة (محلياً فقط)

تحل هذه الخطوة تحديداً مشكلة `CalledProcessError: exit status 2` الناتجة عن اختلاف خيارات سكربت LSTM بين النسخ، وتوحّد ملفات التدريب والتقييم مع نسخة V3 التي جُرّب عليها تدريب LSTM. تُترك ملفات مشروعك الأصلي على Drive دون تغيير.

In [ ]:
from pathlib import Path
import os, shutil, sys, json, zipfile, io, base64, hashlib, ast
if WORK.exists():
    shutil.rmtree(WORK)
def ignore_dirs(directory, names):
    return {n for n in names if n in {'models_dl', 'reports', '.git', '__pycache__', '.ipynb_checkpoints', '.venv', 'venv'}}
shutil.copytree(PROJ, WORK, ignore=ignore_dirs)
assert is_project(WORK), 'النسخة المحلية ناقصة: تحقق من مجلد Drive'
print('✓ تم نسخ كود المشروع والبيانات إلى', WORK)

### 4.1) تثبيت نسخة مدققة من ملفات التدريب فقط في مجلد العمل

هذه الحزمة المضمنة تحتوي على سكربتات AraBERT/LSTM ومرافقها من المشروع الكامل الذي تمت معاينته. لا تُطبّق على Drive ولا تمس ملفات البيانات أو التطبيق.

In [ ]:
import base64, io, zipfile, hashlib, ast
OVERLAY_SHA256 = "c9acfaddca9e8a4ece383a3c1afce271f5fe8418aac5263c3ec00aef0cc9a883"
OVERLAY_B64 = """...
"""
if USE_VERIFIED_TRAINING_CODE:
    blob = base64.b64decode(''.join(OVERLAY_B64.split()))
    assert hashlib.sha256(blob).hexdigest() == OVERLAY_SHA256, 'ملفات التوافق المضمنة تالفة'
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        for name in z.namelist():
            rel = Path(name)
            assert not rel.is_absolute() and '..' not in rel.parts
            dst = WORK / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            dst.write_bytes(z.read(name))
    print('✓ ملفات التدريب V3 المتوافقة: تم تثبيتها محلياً')
else:
    print('تم اختيار كود التدريب الموجود في Drive دون تصحيح')

for rel in ('src/forecasting/train_lstm.py','src/forecasting/series.py',
            'src/training/train_classifier.py','src/training/train_ner.py'):
    ast.parse((WORK / rel).read_text(encoding='utf-8'))
print('✓ تم فحص بناء Python لأربعة سكربتات تدريب')

## 5) إعداد مسارات الحفظ والسجلات ومعاينة واجهات CLI

الأوزان والتقارير تذهب مباشرة إلى **Drive**. تُشغّل أوامر Python داخل نسخة العمل السريعة، مع إظهار سجل كل تدريب كاملاً وإيقاف التنفيذ عند أي فشل بدل إخفائه.

In [ ]:
import os, subprocess, sys, time, json
from pathlib import Path
from datetime import datetime
MODELS_DRIVE = PROJ / 'models_dl'
REPORTS_DRIVE = PROJ / 'reports'
LOG_DIR = REPORTS_DRIVE / 'colab_logs'
for d in (MODELS_DRIVE, REPORTS_DRIVE, LOG_DIR): d.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env.update({
    'LMI_FINAL_DIR': str(WORK / 'FINAL_TRAINING_DATASET'),
    'LMI_MODELS_DIR': str(MODELS_DRIVE),
    'LMI_REPORTS_DIR': str(REPORTS_DRIVE),
    'PYTHONUNBUFFERED': '1',
    'PYTHONIOENCODING': 'utf-8',
    'TOKENIZERS_PARALLELISM': 'false',
    'MPLBACKEND': 'Agg',
    'OMP_NUM_THREADS': str(CPU_THREADS),
})

class TrainingCommandError(RuntimeError): pass

def run_checked(label, cmd, *, max_tail=60):
    name = ''.join(ch if ch.isalnum() or ch in '-_' else '_' for ch in label)
    log = LOG_DIR / f'{name}.log'
    print('\n▶', label, '|', ' '.join(map(str, cmd)), '\nLOG:', log, flush=True)
    t0 = time.monotonic()
    tail = []
    with open(log, 'w', encoding='utf-8') as fp:
        p = subprocess.Popen([str(x) for x in cmd], cwd=WORK, env=env,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, encoding='utf-8', errors='replace', bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            fp.write(line)
            fp.flush()
            tail.append(line.rstrip())
            if len(tail)>max_tail: tail.pop(0)
        rc = p.wait()
    if rc != 0:
        raise TrainingCommandError(f'{label} فشل برمز {rc}. السجل: {log}\n' + '\n'.join(tail))
    print(f'✓ {label} | الزمن {((time.monotonic()-t0)/60):.1f} دقيقة', flush=True)
    return log

for module in ('src.training.train_classifier', 'src.training.train_ner', 'src.forecasting.train_lstm'):
    result = subprocess.run([sys.executable, '-m', module, '--help'], cwd=WORK, env=env,
                            capture_output=True, text=True, encoding='utf-8', errors='replace')
    if result.returncode != 0:
        raise RuntimeError(f'السكربت {module} لا يعمل: {result.stderr[-2000:]}')
    if module.endswith('train_lstm'):
        assert '--families' in result.stdout and '--cpu-threads' in result.stdout and '--as-of' in result.stdout, 'CLI غير متوافق'
    print('✓ CLI:', module)

## 6) فحص البيانات المطلوبة جميعها قبل أي تدريب

يفحص الملفات الأساسية والوسوم والأعمدة. إذا كانت البيانات ناقصة، يظهر اسم الملف بدقة قبل استهلاك وقت التدريب.

In [ ]:
import json
from pathlib import Path
from IPython.display import display
DATA = WORK / 'FINAL_TRAINING_DATASET'
required = [
 'unique_opportunities.csv', 'skills/job_skills.csv',
 'classification_opportunity_type/label_map.json',
 'classification_geographic_scope/label_map.json',
 'ner/label_list.json'
]
for task in ('classification_opportunity_type','classification_geographic_scope'):
    required += [f'{task}/{split}.csv' for split in ('train','validation','test')]
required += [f'ner/{split}.jsonl' for split in ('train','validation','test')]
missing=[x for x in required if not (DATA/x).is_file()]
if missing:
    raise FileNotFoundError('ملفات تدريب مطلوبة غير موجودة في FINAL_TRAINING_DATASET:\n' + '\n'.join(missing))

import sys
sys.path.insert(0, str(WORK))
os.chdir(WORK)
from src.data.loaders import load_classification, load_ner
from src.forecasting.series import build_family
from datetime import date
for task in ('opportunity_type','geographic_scope'):
    splits, labels = load_classification(task, DATA, splits=['train','validation','test'])
    assert all(len(splits[k])>0 for k in splits), 'بيانات فارغة: '+task
    print('✓', task, {k: len(v) for k,v in splits.items()}, 'classes:', len(labels['label2id']))
ner_splits, tagset = load_ner(DATA, splits=['train','validation','test'])
assert all(len(v)>0 for v in ner_splits.values())
print('✓ NER', {k:len(v) for k,v in ner_splits.items()}, 'BIO tags:', len(tagset))
for family in LSTM_FAMILIES:
    info=build_family(family, DATA, as_of=date.fromisoformat(AS_OF))
    print('✓' if info.get('status')=='ok' else '✖', 'LSTM', family,
          'status:', info.get('status'), 'series:', len(info['series'].columns) if 'series' in info else 0)
    if info.get('status') != 'ok':
        raise RuntimeError(f'LSTM {family} لا يملك بيانات كافية: {info}')
print('✓ كل بيانات المهام السبع جاهزة')

## 7) تدريب AraBERT: نوع الفرصة + النطاق الجغرافي + NER

**المعيار:** أفضل أوزان على Validation، ثم تقييم Test لاحقاً بشكل منفصل. يفضّل GPU لتسريع التدريب. لا يتم تخطي نموذج إلا بعد فحص ملفات أوزانه وتوكنزره وسجل التدريب. الحفظ إلى Drive يتم **بعد** اكتمال تدريب كل مهمة ونسخها بأمان.

In [ ]:
import json, shutil, tempfile, time
from pathlib import Path
HF_TMP = Path('/content/lmi_hf_train')
HF_TMP.mkdir(parents=True, exist_ok=True)

def valid_bert(d):
    d = Path(d)
    if not all((d/x).is_file() for x in ('config.json','training_config.json','training_history.csv')):
        return False
    if not any((d/x).is_file() and (d/x).stat().st_size>0 for x in
               ('model.safetensors','pytorch_model.bin','model.safetensors.index.json','pytorch_model.bin.index.json')):
        return False
    return any((d/x).is_file() for x in ('tokenizer.json','vocab.txt','spiece.model','tokenizer.model'))

def safe_publish_dir(src, dst):
    src, dst = Path(src), Path(dst)
    if not src.is_dir(): raise FileNotFoundError(src)
    dst.parent.mkdir(parents=True, exist_ok=True)
    staging = dst.parent / ('.' + dst.name + '_new')
    if staging.exists(): shutil.rmtree(staging)
    shutil.copytree(src, staging)
    if not valid_bert(staging):
        raise RuntimeError(f'لم تكتمل ملفات النموذج عند الحفظ: {staging}')
    backup = dst.parent / ('.' + dst.name + '_backup')
    if backup.exists(): shutil.rmtree(backup)
    if dst.exists(): dst.rename(backup)
    try:
        staging.rename(dst)
    except Exception:
        if backup.exists() and not dst.exists(): backup.rename(dst)
        raise
    print('✓ تم حفظ النموذج بأمان في Drive:', dst)

bert_done = {}
for task in ARABERT_TASKS:
    dest = MODELS_DRIVE / ('ner' if task=='ner' else f'classification/{task}')
    if ARABERT_MODE=='missing' and valid_bert(dest):
        print('↷ AraBERT', task, ': نموذج كامل موجود مسبقاً؛ تم التخطي')
        bert_done[task]='existing'
        continue
    out = HF_TMP / task
    if out.exists(): shutil.rmtree(out)
    cmd=[sys.executable, '-m', 'src.training.train_ner'] if task=='ner' else [sys.executable, '-m', 'src.training.train_classifier', '--task', task]
    cmd += ['--data_dir', str(DATA), '--out', str(out)]
    run_checked('arabert_'+task, cmd)
    if not valid_bert(out): raise RuntimeError('اكتمل الأمر دون ملفات BERT كافية: '+task)
    safe_publish_dir(out, dest)
    bert_done[task]='trained'
print('✓ AraBERT tasks:', bert_done)

## 8) تدريب LSTM للأربع عائلات — تدريب حقيقي على CPU

تُشغَّل كل عائلة مستقلة ويسجل سجلها. تستخدم السلاسل الشهرية حصة الطلب، تقسيم زمني Train/Validation/Test، ونافذة 6 أشهر؛ تتعلم على **MSE** وتُقارن MAE وRMSE مع Naive وMA-3. لا تنس أن `LSTM_MODE="force"` يعيد تدريبها حتى إن كانت أوزانها موجودة.

In [ ]:
import json, torch

def valid_lstm(family):
    model_dir=MODELS_DRIVE/'lstm'/family
    eval_dir=REPORTS_DRIVE/'evaluation'/('lstm_'+family)
    needed_m=['lstm.pt','config.json','scaler.json']
    needed_r=['metrics.json','test_predictions.csv','trend_forecast.csv','future_forecast.csv']
    if not all((model_dir / k).is_file() and (model_dir / k).stat().st_size>0 for k in needed_m): return False
    if not all((eval_dir / k).is_file() and (eval_dir / k).stat().st_size>0 for k in needed_r): return False
    try:
        cfg=json.loads((model_dir/'config.json').read_text(encoding='utf-8'))
        met=json.loads((eval_dir/'metrics.json').read_text(encoding='utf-8'))
        assert cfg['family']==family and met.get('trained') is True
        assert 'mae' in met['test']['lstm']
        from src.forecasting.train_lstm import make_model
        m=make_model(cfg['hidden_size'], 1, cfg['dropout'])
        m.load_state_dict(torch.load(model_dir/'lstm.pt', map_location='cpu', weights_only=True))
        return True
    except Exception as ex:
        print('ملاحظة: ملفات LSTM غير مكتملة أو غير متوافقة:', family, repr(ex))
        return False

lstm_done = {}
for family in LSTM_FAMILIES:
    if LSTM_MODE=='missing' and valid_lstm(family):
        print('↷ LSTM', family, ': تم التخطي بعد فحص الأوزان والتقارير')
        lstm_done[family]='existing'
        continue
    cmd=[sys.executable, '-m', 'src.forecasting.train_lstm',
         '--families', family, '--data_dir', str(DATA),
         '--as-of', AS_OF, '--cpu-threads', str(CPU_THREADS)]
    run_checked('lstm_'+family, cmd)
    if not valid_lstm(family):
        raise RuntimeError('عاد أمر التدريب دون نموذج LSTM كامل: '+family+
                           '. راجع reports/colab_logs/lstm_'+family+'.log')
    lstm_done[family]='trained'
print('✓ LSTM tasks:', lstm_done)

## 9) تقييم AraBERT على مجموعة Test (بعد التدريب فقط)

هذه مرحلة منفصلة عن اختيار النموذج. يتم حساب مقاييس Silver Test من البيانات المحفوظة فعلياً. إذا كان ملف تقييم قديم موجوداً، يُعاد توليده في هذه الجلسة.

In [ ]:
for task in ('opportunity_type','geographic_scope'):
    dest=MODELS_DRIVE/'classification'/task
    assert valid_bert(dest), 'لا يوجد نموذج AraBERT متكامل لمهمة: '+task
    run_checked('evaluate_'+task, [sys.executable, '-m','src.evaluation.evaluate_classifier',
                                   '--task', task, '--model_dir', str(dest),
                                   '--data_dir', str(DATA),
                                   '--out',str(REPORTS_DRIVE/'evaluation'/('classification_'+task))])
assert valid_bert(MODELS_DRIVE/'ner'), 'نموذج NER غير مكتمل'
run_checked('evaluate_ner', [sys.executable, '-m', 'src.evaluation.evaluate_ner',
                             '--model_dir', str(MODELS_DRIVE/'ner'),
                             '--data_dir', str(DATA),
                             '--out', str(REPORTS_DRIVE/'evaluation'/'ner')])
print('✓ اكتمل تقييم AraBERT على Test')

## 10) عرض ملخص النتائج والتحقق من حفظ كل النماذج السبعة

لا نعتبر أي مهمة منتهية بمجرد انتهاء الأمر؛ يجب وجود النموذج القابل للتحميل وتقرير تقييم حقيقي. جدول LSTM يعرض MAE للنموذج وخطي الأساس (الأصغر أفضل)، ولا يدّعي تفوق LSTM إن لم يحقق ذلك.

In [ ]:
import pandas as pd, json
from IPython.display import display
for task in ARABERT_TASKS:
    d=MODELS_DRIVE/('ner' if task=='ner' else f'classification/{task}')
    assert valid_bert(d), 'نموذج AraBERT غير جاهز: '+task
lstm_rows=[]
for family in LSTM_FAMILIES:
    assert valid_lstm(family), 'نموذج LSTM ناقص: '+family
    p=REPORTS_DRIVE/'evaluation'/('lstm_'+family)/'metrics.json'
    r=json.loads(p.read_text(encoding='utf-8'))
    lstm_rows.append({'family':family, 'series':r.get('series'),
                      'test_period':r.get('test_period'),
                      'LSTM_MAE':r['test']['lstm']['mae'],
                      'Naive_MAE':r['test']['naive']['mae'],
                      'MA3_MAE':r['test']['moving_avg_3']['mae']})
print('نتائج LSTM (Silver/Test زمنية):')
display(pd.DataFrame(lstm_rows))

bert_rows=[]
for task, folder in [('opportunity_type','classification_opportunity_type'),
                     ('geographic_scope','classification_geographic_scope'), ('ner','ner')]:
    p=REPORTS_DRIVE/'evaluation'/folder/'metrics.json'
    if not p.is_file(): raise FileNotFoundError('نتيجة التقييم مفقودة: '+str(p))
    obj=json.loads(p.read_text(encoding='utf-8'))
    bert_rows.append({'task':task,'metric': 'entity_f1' if task=='ner' else 'macro_f1',
                      'value':obj.get('entity_f1') if task=='ner' else obj.get('macro_f1'),
                      'label_quality':obj.get('label_quality','silver')})
print('نتائج AraBERT على Test:')
display(pd.DataFrame(bert_rows))
print('✓ تم التحقق من وجود أوزان النماذج السبعة وتقاريرها في Drive')

## 11) تشغيل اختبارات المشروع (اختياري لكن مفعّل)

الاختبارات تفحص وظائف الكود باستخدام بيانات اصطناعية للاختبار فقط، **ولا تعني تحقق دقة النماذج على بيانات حقيقية**.

In [ ]:
if RUN_TESTS:
    testfile = WORK / 'tests/test_dl_system.py'
    if testfile.exists():
        run_checked('project_pytest', [sys.executable, '-m', 'pytest', '-q',
                    'tests/test_dl_system.py', '-p', 'no:cacheprovider'])
    else:
        print('لا يوجد ملف tests/test_dl_system.py في هذا المشروع؛ تم تجاوز الاختبارات')
else:
    print('الاختبارات معطّلة في الإعدادات')

## 12) تصدير النتائج للكمبيوتر (اختياري)

إذا فعّلت `PACKAGE_FOR_WINDOWS=True` فسينشئ ملفي ZIP في MyDrive، ويمكن فكهما في مجلد المشروع على Windows. لا يلزم تشغيل هذا الجزء لتدريب النماذج.

In [ ]:
if PACKAGE_FOR_WINDOWS:
    run_checked('package_results', [sys.executable, 'scripts/package_results.py', '--out', str(MY), '--fp16'])
    print('✓', MY/'lmi_models.zip', MY/'lmi_reports.zip')
else:
    print('التصدير معطّل؛ الأوزان والتقارير محفوظة مباشرة في Drive')

## 13) تشغيل واجهة Streamlit (اختياري)

عند التفعيل يبدأ تطبيق Streamlit ويفتحه باستخدام Cloudflare إن توفّر اتصال. هذا **غير مطلوب** للتدريب، وقد يفشل النفق بسبب سياسات الشبكة دون أن يؤثر على النماذج المحفوظة.

In [ ]:
if RUN_STREAMLIT:
    import re, urllib.request, subprocess, time, threading
    cloudflared = Path('/content/cloudflared')
    if not cloudflared.is_file():
        urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
        cloudflared.chmod(0o755)
    app = subprocess.Popen([sys.executable,'-m','streamlit','run', str(WORK/'app_dl/main.py'),
                            '--server.port','8501','--server.headless','true'],
                           cwd=WORK,env=env, stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
    time.sleep(7)
    if app.poll() is not None: raise RuntimeError('فشل تشغيل تطبيق Streamlit؛ تحقق من ملفات التطبيق')
    tunnel = subprocess.Popen([str(cloudflared),'tunnel','--url','http://127.0.0.1:8501','--no-autoupdate'],
                              stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    link=None
    for i,line in enumerate(tunnel.stdout):
        m=re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com',line)
        if m: link=m.group(0);break
        if i>120: break
    print('رابط التطبيق:', link or 'تعذر الحصول على رابط؛ التدريب محفوظ بالفعل')
else:
    print('التطبيق غير مفعّل؛ اكتمل التدريب والتقييم دون الحاجة إليه')

## انتهى — أماكن المخرجات وصدق النتائج

- أوزان AraBERT: `MyDrive/LMI_PROJECT/models_dl/classification/{opportunity_type,geographic_scope}/` و`models_dl/ner/`.
- أوزان LSTM: `MyDrive/LMI_PROJECT/models_dl/lstm/{skills,specializations,job_titles,governorates}/`.
- نتائج الاختبار: `MyDrive/LMI_PROJECT/reports/evaluation/`.
- السجلات الكاملة: `MyDrive/LMI_PROJECT/reports/colab_logs/`.

**للتوضيح العلمي:** AraBERT يقيَّم غالباً على Silver Labels، وليس بيانات Gold موثقة بشرية. نتائج LSTM تعتمد على الحصة الشهرية، وتقييم اختبارها هو تنبؤ خطوة واحدة، بينما التوقع لستة أشهر بعد آخر بيانات تكراري. لا تحذف ملفات المشروع الأصلية من Drive.